## Beaty Informatics x MDS Capstone: Proposal

### Executive Summary
##### A brief overview of the problem, your proposed approach, and the expected outcome

Taxonomy, including species names and ranks, is constantly changing with the advent of new scientific methods. Curators at the Beaty Biodiversity Museum have to keep track of all of these changes in order to keep the collections current, but finding and evaluating information can be time consuming. We propose creating a web app which synthesizes current information online regarding species name synonyms and taxonomy and provides it to curators, who can then decide which designation they'd like to incorporate into the museum's database. 


### Introduction
##### Describe the problem in your own words, Explain why it is important, Provide relevant background
- Taxonomy refers to naming, describing, and classifying organisms into hierarchical groups based on shared characteristics and evolutionary relationships.
- Museum curators are responsible for keeping organized, up-to-date, searchable collections for researchers and experts to use, with  up-to-date taxonomic classifications and names. 
- Individual species names can change over time, introducing synonyms, basionyms, variants, and subspecies which can get in the way of curator's overall goals. 
- taxonomic classifications (e.g. family, order) can also change, introducing further uncertainty into where organisms should physically and digitally be stored. 
- Databases such as GBIF (Global Biodiversity Information Facility) and GenBank, as well as more taxa-specific sites such as Mushroom Observer and BugGuide, publish updated species names, name changes, and taxonomic classification changes

Refine the problem into specific objectives
- When a specimen comes into the museums, curators are responsible for identifying it and classifying it correctly. When they are unsure if the name or classification has changed, that's where we step in. By making multiple API calls to different sites, both taxa specific and not, we can provide a list of species synonyms which are online. 
- Curators can select which sites they prefer to refer to, and see species synonyms.
- Synonyms are the first step of our project; we hope to expand into species taxonomy and even ranges. 

Describe the final data product (e.g., dashboard, model, pipeline)
- The final data product is a web app which synthesizes online information with a search. 

### Data & Exploratory Data Analysis (EDA)
Provide a clear description and initial exploration of your data:
- Data source(s), size, and structure

Data sources vary by taxonomic group, with some sources in common between them (for example, GBIF, GenBank). API calls return JSON-formatted species synonyms, variants, and subspecies. The output is structured as synonyms being the key, and variants / subspecies are the values. Size varies by species - names could have 0 variants, or dozens. Certain species may have no synonyms, some have plenty. 

- Definition of the observational unit

An observation unit, in this case, could be considered either species, or their name synonyms. 

- Key variables/features


Include tables or visualizations showing data quality issues (e.g., missing values, imbalance) and patterns or signals relevant to your task. Also, discuss limitations of the data and potential challenges and their implications. Avoid including raw code in the report. Instead, use a pipeline to generate figures and results.

In [1]:
import requests
import pandas as pd
import sys
sys.path.insert(0, "..")

from scripts.APIs.GBIF import get_gbif_synonyms

In [4]:
# we should probably change this to be not JUST the gbif function. 
result = get_gbif_synonyms("Amanita muscaria")

def display_synonyms_markdown(synonyms: dict, query: str):
    lines = [f"## Synonyms for *{query}*\n"]
    for species, variants in synonyms.items():
        lines.append(f"- **{species}**")
        for v in variants:
            lines.append(f"  - *{v}*")
    from IPython.display import Markdown, display
    display(Markdown("\n".join(lines)))

display_synonyms_markdown(result, "Amanita muscaria")

## Synonyms for *Amanita muscaria*

- **Agaricus aureolus**
- **Agaricus imperialis**
- **Agaricus muscarius**
  - *Agaricus muscarius formosus*
  - *Agaricus muscarius puella*
  - *Agaricus muscarius sanguineus*
- **Agaricus nobilis**
- **Agaricus pseudoaurantiacus**
- **Agaricus puellus**
- **Amanita aureola**
- **Amanita circinnata**
- **Amanita formosa**
- **Amanita muscaria**
  - *Amanita muscaria aureola*
  - *Amanita muscaria beglyanovae*
  - *Amanita muscaria eu-umbrina*
  - *Amanita muscaria europaea*
  - *Amanita muscaria flavivolvata*
  - *Amanita muscaria formosa*
  - *Amanita muscaria guessowii*
  - *Amanita muscaria puella*
  - *Amanita muscaria vaginata*
  - *Amanita muscaria americana*
  - *Amanita muscaria hercynica*
  - *Amanita muscaria minor*
  - *Amanita muscaria sudedica*
  - *Amanita muscaria umbrina*
  - *Amanita muscaria alba*
  - *Amanita muscaria alpha*
  - *Amanita muscaria coccinea*
  - *Amanita muscaria fuligineoverrucosa*
  - *Amanita muscaria sanguinea*
  - *Amanita muscaria speciosa*
  - *Amanita muscaria tomentosa*
  - *Amanita muscaria vulgaris*
- **Amanita puella**
- **Amanitaria muscaria**
- **Hypophyllum muscarium**
- **Venenarius muscarius**

In [5]:
df = pd.DataFrame([
    {"Species": species, "# Variants": len(variants), "Variants": ", ".join(variants) or "—"}
    for species, variants in result.items()
])
df

,Species,# Variants,Variants
0,Agaricus aureolus,0,—
1,Agaricus imperialis,0,—
2,Agaricus muscarius,3,"Agaricus muscarius formosus, Agaricus muscariu..."
3,Agaricus nobilis,0,—
4,Agaricus pseudoaurantiacus,0,—
5,Agaricus puellus,0,—
6,Amanita aureola,0,—
7,Amanita circinnata,0,—
8,Amanita formosa,0,—
9,Amanita muscaria,22,"Amanita muscaria aureola, Amanita muscaria beg..."


### Data Science Approach

Our approach is structured around thoroughness, user efficiency, and clarity. Our main task is collecting information from several external databases, combining it, and showing it clearly so curators can make a well-informed decision.

Curators work with resources written across many different eras and often need to reconcile older and newer terminology (e.g. species name). The app gives them a full picture of name changes without having to compile it themselves by searching individual databases. This matters not just for curators, but also for visiting experts: if the collection uses a different name than what an expert is searching for, they may not be able to find the specimen at all.

**How it works**

A curator starts with a species name they want to check, for example *Amanita muscaria*. They enter it into the app, and the following happens:

1. The app queries each configured database (GBIF, GenBank, Mushroom Observer, iNaturalist, etc.) and collects what each one knows about that name, including synonyms, and metadata like the year a name was published.
2. Results from all databases are combined and shown to the user in a useful way. As each database returns very different result formats, consolidating them into a unified summary is one of the main challenges of this project. This is the area we will continuously explore and evaluate as the project evolves.
3. The curator sees a summary table of all candidate names, possibly some metrics alongside those names, and can decide which name to use for their specimen. Some useful metrics might include:
    - How many databases list this name (more databases = more widely recognized)
    - How recently the name was published or last updated, where that information is available
   How these metrics are weighted or combined is still an open question -- they may be displayed separately or aggregated in some way.

### Evaluation & Success Criteria

Our evaluation focuses on how useful and reliable the information we surface is. We measure success by how well the app helps curators make informed decisions.

The app is considered successful if it can accurately retrieve relevant names from the configured sources and present them in a way that genuinely helps curators make a determination. The key word is *helps*: the tool is not making the decision. It is giving the curator better information so they can make the final call themselves. The curator remains the expert.



Given these priorities, our evaluation will focus on whether the information presented by the app is complete and clear enough to support the curator's final decision.

### Timeline
Provide a high-level timeline outlining key milestones, expected deliverables, and the overall sequence of work.
